##  Installation Set up

In [1]:
!pip install llama-index llama-index-embeddings-huggingface pymupdf
!pip install nest_asyncio

## Install Libraries

In [21]:
import nest_asyncio
nest_asyncio.apply()

from llama_index.core import VectorStoreIndex, Document, Settings, get_response_synthesizer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.query_engine import RetrieverQueryEngine
import fitz  # PyMuPDF
import time
from llama_index.core import Settings
Settings.llm = None

LLM is explicitly disabled. Using MockLLM.


## Upload Document



In [8]:
# Uploaded file
pdf_path = "/content/sample_data/sample_contract_1.pdf"
doc = fitz.open(pdf_path)
text = "\\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the contract.")

Extracted 315 words from the contract.


In [22]:
# Sentence splitter (can also use TokenTextSplitter or CharacterTextSplitter)
text_splitter = SentenceSplitter(chunk_size=50, chunk_overlap=50)

In [23]:
# Turn raw text into a list of Document objects
documents = [Document(text=text)]

In [24]:
# Convert into nodes (smaller chunks)
nodes = text_splitter.get_nodes_from_documents(documents)

## Different Embedding Models

In [27]:
embedding_models = {
    "MiniLM-L6-v2": "sentence-transformers/all-MiniLM-L6-v2",
    "BGE-small-en": "BAAI/bge-small-en-v1.5",
    "E5-small-v2": "intfloat/e5-small-v2"
}

## LOOP through different models

In [32]:
query = "What are the penalties for late payments?"

results = {}

for model_name, model_path in embedding_models.items():
    print(f"Testing Embedding Model: {model_name}")

    # Configure the embedding model
    embed_model = HuggingFaceEmbedding(model_name=model_path)
    Settings.embed_model = embed_model

    # Build the index
    start_time = time.time()

    # Build index
    index = VectorStoreIndex(nodes)

    retriever = index.as_retriever(similarity_top_k=2)
    query_engine = RetrieverQueryEngine.from_args(retriever=retriever)

    # Run the query
    response = query_engine.query(query)
    end_time = time.time()

    # Store results
    results[model_name] = {
        "response": str(response),
        "time": round(end_time - start_time, 2)
    }


Testing Embedding Model: MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Testing Embedding Model: BGE-small-en


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Testing Embedding Model: E5-small-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Final Result

In [33]:
for model, result in results.items():
    print(f"-------------------------------------------")
    print(f"Model: {model}")
    print(f"Retrieval Time: {result['time']} seconds")
    print(f"Response: {result['response']}")
    print(f"-------------------------------------------")

-------------------------------------------
Model: MiniLM-L6-v2
Retrieval Time: 0.45 seconds
Response: Context information is below.
---------------------
Payment terms are
net 30 days from receipt of invoice.
2.3 Late payments shall bear interest at the rate of 1.5% per month from the due date until paid in full.
3.

4.2 Refunds are issued at the sole discretion of Service Provider and will be processed within 30 days
of approval.
\n4.3 No refunds will be issued for completed projects that meet the specifications outlined in Exhibit A.
5.
---------------------
Given the context information and not prior knowledge, answer the query.
Query: What are the penalties for late payments?
Answer: 
-------------------------------------------
-------------------------------------------
Model: BGE-small-en
Retrieval Time: 0.84 seconds
Response: Context information is below.
---------------------
Payment terms are
net 30 days from receipt of invoice.
2.3 Late payments shall bear interest at the ra